In [2]:
import os
print("JAVA_HOME:", os.environ.get('JAVA_HOME'))


JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64


In [3]:
from pathlib import Path
import os, sys, subprocess, importlib.metadata
IS_COLAB = Path('/content').is_dir() and 'google.colab' in sys.modules
if IS_COLAB:
    ROOT = Path('/content/masar-modern-data-engineering')
    if not ROOT.exists():
        subprocess.run(['git','clone','https://github.com/almiyead-rgb/masar-modern-data-engineering.git',str(ROOT)], check=True)
    pins = {'pyspark':'3.5.8','delta-spark':'3.3.3','py4j':'0.10.9.9'}
    missing = []
    for package, version in pins.items():
        try: observed = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError: observed = None
        if observed != version: missing.append(f'{package}=={version}')
    if missing:
        subprocess.run([sys.executable,'-m','pip','install','--quiet',*missing],check=True)
    candidates = list(Path('/usr/lib/jvm').glob('*17*'))
    if not any((p/'bin/java').is_file() for p in candidates):
        subprocess.run(['apt-get','update','-qq'],check=True)
        subprocess.run(['apt-get','install','-y','-qq','openjdk-17-jre-headless'],check=True)
        candidates = list(Path('/usr/lib/jvm').glob('*17*'))
    os.environ['JAVA_HOME'] = str(next(p for p in candidates if (p/'bin/java').is_file()))
    os.chdir(ROOT)
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p/'course.json').is_file()),None)
    if ROOT is None: raise FileNotFoundError('Open from the repository root; see docs/SETUP.md')
print('Repository:', ROOT)
print('Python:', sys.version.split()[0])

Repository: /home/AlbandriAAlotaibi/masar-modern-data-engineering
Python: 3.11.9


In [4]:
from pathlib import Path
import sys, json, tempfile
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src" / "masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the complete course repository.")
sys.path.insert(0, str(ROOT / "src"))
(ROOT / "outputs").mkdir(exist_ok=True)
RUN = Path(tempfile.mkdtemp(prefix="day01_", dir=ROOT / "outputs"))
print("Inspect the source and assumptions before running the next native section.")

Inspect the source and assumptions before running the next native section.


In [5]:
from masar.sources import verify_manifest, load_sources, profile_sources
SOURCE = ROOT / "data" / "masar-small-v1"
manifest = verify_manifest(SOURCE)
feeds = load_sources(SOURCE)
print("Dataset:", manifest["label"])
print("Verified manifest files:", len(manifest["files"]))
print(json.dumps({name: len(rows) for name, rows in feeds.items()}, indent=2))
print("Trip columns:", list(feeds["trips"][0]))
print("First synthetic event:", json.dumps(feeds["gps_events"][0], ensure_ascii=False))

Dataset: MASAR_SMALL_V1
Verified manifest files: 10
{
  "trips": 72,
  "drivers": 6,
  "gps_events": 216
}
Trip columns: ['trip_id', 'driver_id', 'city', 'start_ts', 'end_ts', 'fare_sar', 'distance_km']
First synthetic event: {"city": "Riyadh", "event_id": "SYN_E0001_0", "event_ts": "2026-06-01T06:00:00+03:00", "location": {"lat": 24.7, "lon": 46.7}, "synthetic": true, "trip_id": "SYN_T0001"}


In [6]:
result = profile_sources(SOURCE)
print(json.dumps(result["profile"], indent=2))
print(json.dumps(result["relations"], indent=2))
print(json.dumps(result["city_profile"], indent=2))

{
  "trips": {
    "rows": 72,
    "key": "trip_id",
    "duplicate_key_groups": 0,
    "duplicate_excess_rows": 0,
    "missing_top_level_fields": {
      "city": 0,
      "distance_km": 0,
      "driver_id": 0,
      "end_ts": 0,
      "fare_sar": 0,
      "start_ts": 0,
      "trip_id": 0
    }
  },
  "drivers": {
    "rows": 6,
    "key": "driver_id",
    "duplicate_key_groups": 0,
    "duplicate_excess_rows": 0,
    "missing_top_level_fields": {
      "driver_id": 0,
      "driver_rating": 0,
      "vehicle_type": 0
    }
  },
  "gps_events": {
    "rows": 216,
    "key": "event_id",
    "duplicate_key_groups": 0,
    "duplicate_excess_rows": 0,
    "missing_top_level_fields": {
      "city": 0,
      "event_id": 0,
      "event_ts": 0,
      "location": 0,
      "synthetic": 0,
      "trip_id": 0
    }
  }
}
{
  "trips_without_driver": 0,
  "events_without_trip": 0,
  "events_with_invalid_coordinates": 0
}
{
  "original_labels": {
    " dammam ": 3,
    " jeddah ": 3,
    " riyad

In [7]:
if not all(result["checks"].values()):
    raise AssertionError(result["checks"])
output = RUN / "source_inspection.json"
output.write_text(json.dumps(result, ensure_ascii=False, sort_keys=True, indent=2) + "\n", encoding="utf-8")
print(json.dumps(result["checks"], indent=2))
print("PASS: source inspection only")
print("Saved:", output.name)
print("Source inspection complete. Continue to the Bronze section.")

{
  "source_counts": true,
  "unique_base_keys": true,
  "base_top_level_complete": true,
  "valid_links_and_coordinates": true,
  "three_events_per_trip": true,
  "base_city_set": true
}
PASS: source inspection only
Saved: source_inspection.json
Source inspection complete. Continue to the Bronze section.


In [8]:
import os
print("JAVA_HOME:", os.environ.get('JAVA_HOME'))

JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64


In [9]:
from pathlib import Path
import sys, json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src/masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open the notebook from within the complete course repository")
sys.path.insert(0, str(ROOT / "src"))
SOURCE = ROOT / "data/masar-small-v1"
from masar.runtime import inspect_environment, require_environment, start_spark
print(json.dumps(inspect_environment(), indent=2))

{
  "scope": "DEPENDENCY_PREFLIGHT_ONLY",
  "python": "3.11.9",
  "java": "openjdk version \"17.0.20\" 2026-07-21",
  "java_major": 17,
  "packages": {
    "pyspark": {
      "required": "3.5.8",
      "observed": "3.5.8"
    },
    "delta-spark": {
      "required": "3.3.3",
      "observed": "3.3.3"
    },
    "py4j": {
      "required": "0.10.9.9",
      "observed": "0.10.9.9"
    }
  },
  "status": "DEPENDENCIES_PRESENT_ENGINE_NOT_TESTED",
  "issues": [],
  "engine_executed": false
}


In [10]:
require_environment()
from masar.workspace import new_workspace, require_fixed_dataset, write_json, workspace_path
require_fixed_dataset(SOURCE)
WORK = new_workspace(ROOT, "day01_bronze")
print("Workspace:", WORK.relative_to(ROOT))

Workspace: outputs/day01_bronze_lalzg7ih


In [11]:
from masar.bronze import raw_frame, ingest_feed, verify_bronze
# Functions are imported without starting Spark. The next cell executes the lab.
print("Bronze functions loaded. The next cell writes and reads the Delta tables.")

Bronze functions loaded. The next cell writes and reads the Delta tables.


In [12]:
from masar.workspace import record_bronze_success
spark = start_spark(WORK)
print("Spark:", spark.version)

https://repo.maven.apache.org/maven2 added as a remote repository with the name: repo-1


:: loading settings :: url = jar:file:/home/AlbandriAAlotaibi/masar-modern-data-engineering/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/AlbandriAAlotaibi/.ivy2/cache
The jars for the packages stored in: /home/AlbandriAAlotaibi/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-072713f2-cc24-4d1c-ad92-d217fa08988b;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.3 in central
	found io.delta#delta-storage;3.3.3 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 20393ms :: artifacts dl 52ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.3.3 from central in [default]
	io.delta#delta-storage;3.3.3 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      d

Spark: 3.5.8


In [13]:
spark.stop()

In [14]:
import os
print("JAVA_HOME:", os.environ.get('JAVA_HOME'))

JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64


In [15]:
from masar.workspace import record_bronze_success
spark = start_spark(WORK)
try:
    print("Spark:", spark.version)
    raw_trips = raw_frame(spark, SOURCE, "trips")
    raw_trips.printSchema()
    raw_trips.show(3, truncate=False)
    print("Source trips:", raw_trips.count())
    for feed in ("trips", "drivers", "gps_events"):
        print(ingest_feed(spark, SOURCE, WORK, feed, "base_001"))
    print(ingest_feed(spark, SOURCE, WORK, "trips", "replay_002"))
    report = verify_bronze(spark, SOURCE, WORK)
    record_bronze_success(ROOT, WORK)
    print(json.dumps({"counts": report["counts"], "checks": report["checks"]}, indent=2))
    print("Retained:", (WORK / "reports/bronze.json").relative_to(ROOT))
finally:
    spark.stop()

Spark: 3.5.8
root
 |-- trip_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- start_ts: string (nullable = true)
 |-- end_ts: string (nullable = true)
 |-- fare_sar: string (nullable = true)
 |-- distance_km: string (nullable = true)

+---------+---------+------+-------------------------+-------------------------+--------+-----------+
|trip_id  |driver_id|city  |start_ts                 |end_ts                   |fare_sar|distance_km|
+---------+---------+------+-------------------------+-------------------------+--------+-----------+
|SYN_T0001|SYN_D001 |Riyadh|2026-06-01T06:00:00+03:00|2026-06-01T06:08:00+03:00|18.00   |3.50       |
|SYN_T0002|SYN_D002 |Riyadh|2026-06-01T08:00:00+03:00|2026-06-01T08:11:00+03:00|19.25   |3.85       |
|SYN_T0003|SYN_D001 |Riyadh|2026-06-01T10:00:00+03:00|2026-06-01T10:14:00+03:00|20.50   |4.20       |
+---------+---------+------+-------------------------+-------------------------+--------+---

In [16]:
spark.stop()

In [17]:
from pathlib import Path
import sys, json, tempfile
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src" / "masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the complete course repository.")
sys.path.insert(0, str(ROOT / "src"))
(ROOT / "outputs").mkdir(exist_ok=True)
RUN = Path(tempfile.mkdtemp(prefix="day01_", dir=ROOT / "outputs"))
print("Inspect the source and assumptions before running the next native section.")

Inspect the source and assumptions before running the next native section.


In [18]:
from decimal import Decimal
from masar.cost import DEFAULTS, evaluate_cost, serializable, cost_report
print(json.dumps(DEFAULTS, indent=2))
result = cost_report()
print(json.dumps(result["base"], indent=2))

{
  "days": "30",
  "hours_per_day": "24",
  "cores": "4",
  "price_per_core_hour": "0.50",
  "storage_gib": "100",
  "storage_price_per_gib_month": "0.20",
  "work_hours_per_day": "2",
  "startup_hours_per_day": "0.25",
  "scheduled_extra_monthly": "30",
  "always_on_extra_monthly": "0"
}
{
  "unit": "TU (hypothetical teaching units; not currency)",
  "always_on_hours": "720",
  "scheduled_hours": "67.50",
  "always_on_compute": "1440.00",
  "scheduled_compute": "135.0000",
  "storage_each": "20.00",
  "always_on_total": "1460.00",
  "scheduled_total": "185.0000",
  "difference": "1275.0000",
  "reduction_fraction": "0.8732876712328767123287671233",
  "break_even_work_hours_per_day": "23.25"
}


In [19]:
print("Work h/day | Always-on TU | Scheduled TU | Difference TU")
for row in result["sensitivity"]:
    print(f"{row['work_hours_per_day']:>10} | {row['always_on_total']:>12} | {row['scheduled_total']:>12} | {row['difference']:>13}")

Work h/day | Always-on TU | Scheduled TU | Difference TU
         0 |      1460.00 |        50.00 |       1410.00
         1 |      1460.00 |     125.0000 |     1335.0000
         2 |      1460.00 |     185.0000 |     1275.0000
         4 |      1460.00 |     305.0000 |     1155.0000
         8 |      1460.00 |     545.0000 |      915.0000
        12 |      1460.00 |     785.0000 |      675.0000
        18 |      1460.00 |    1145.0000 |      315.0000
        23 |      1460.00 |    1445.0000 |       15.0000
     23.25 |      1460.00 |    1460.0000 |        0.0000
     23.75 |      1460.00 |    1490.0000 |      -30.0000


In [20]:
base = evaluate_cost(DEFAULTS)
checks = {
    "base_totals": (base["always_on_total"], base["scheduled_total"]) == (Decimal("1460"), Decimal("185")),
    "break_even": evaluate_cost({**DEFAULTS, "work_hours_per_day": "23.25"})["difference"] == 0,
    "counterexample": evaluate_cost({**DEFAULTS, "work_hours_per_day": "23.75"})["difference"] < 0,
    "zero_rate": evaluate_cost({**DEFAULTS, "price_per_core_hour": "0"})["break_even_work_hours_per_day"] is None,
}
try:
    evaluate_cost({**DEFAULTS, "cores": "-1"})
except ValueError:
    checks["negative_input_rejected"] = True
else:
    checks["negative_input_rejected"] = False
if not all(checks.values()):
    raise AssertionError(checks)
result["checks"] = checks
print(json.dumps(checks, indent=2))

{
  "base_totals": true,
  "break_even": true,
  "counterexample": true,
  "zero_rate": true,
  "negative_input_rejected": true
}


In [21]:
output = RUN / "cost_model_result.json"
output.write_text(json.dumps(result, sort_keys=True, indent=2) + "\n", encoding="utf-8")
print("PASS: hypothetical cost arithmetic only")
print("Saved:", output.name)
print("Cost arithmetic complete. Continue to the measured Spark comparison.")

PASS: hypothetical cost arithmetic only
Saved: cost_model_result.json
Cost arithmetic complete. Continue to the measured Spark comparison.


In [22]:
from pathlib import Path
import sys, json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src/masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open the notebook from within the complete course repository")
sys.path.insert(0, str(ROOT / "src"))
SOURCE = ROOT / "data/masar-small-v1"
from masar.runtime import inspect_environment, require_environment, start_spark
print(json.dumps(inspect_environment(), indent=2))

{
  "scope": "DEPENDENCY_PREFLIGHT_ONLY",
  "python": "3.11.9",
  "java": "openjdk version \"17.0.20\" 2026-07-21",
  "java_major": 17,
  "packages": {
    "pyspark": {
      "required": "3.5.8",
      "observed": "3.5.8"
    },
    "delta-spark": {
      "required": "3.3.3",
      "observed": "3.3.3"
    },
    "py4j": {
      "required": "0.10.9.9",
      "observed": "0.10.9.9"
    }
  },
  "status": "DEPENDENCIES_PRESENT_ENGINE_NOT_TESTED",
  "issues": [],
  "engine_executed": false
}


In [23]:
require_environment()
from masar.workspace import completed_bronze_workspace, require_fixed_dataset
require_fixed_dataset(SOURCE)
WORK = completed_bronze_workspace(ROOT)
spark = start_spark(WORK)
print("Workspace:", WORK.relative_to(ROOT))

Workspace: outputs/day01_bronze_lalzg7ih


In [24]:
from masar.benchmark import benchmark
try:
    report = benchmark(spark, SOURCE, WORK, repetitions=4)
    print(json.dumps(report["expected_and_observed_aggregate"], indent=2))
    print(json.dumps(report["measurements"], indent=2))
    print("Plans:", report["plans"])
    print("Evidence:", (WORK / "reports/benchmark.json").relative_to(ROOT))
finally:
    spark.stop()

{
  "rows": 72,
  "nonnull_fares": 72,
  "fare_total": "1794.60"
}
{
  "csv": {
    "samples_s": [
      1.279472240000132,
      0.12166447799995694,
      0.14228317099991727,
      0.12130510099996172
    ],
    "median_s": 0.1319738244999371,
    "min_s": 0.12130510099996172,
    "max_s": 1.279472240000132
  },
  "delta_v0": {
    "samples_s": [
      3.849838054999964,
      0.8242476909999823,
      0.9123258139998143,
      1.266349893000097
    ],
    "median_s": 1.0893378534999556,
    "min_s": 0.8242476909999823,
    "max_s": 3.849838054999964
  }
}
Plans: {'csv': 'reports/plans/csv.txt', 'delta_v0': 'reports/plans/delta_v0.txt'}
Evidence: outputs/day01_bronze_lalzg7ih/reports/benchmark.json


In [25]:
# DAY01_HANDOFF_V2: retain the pointer and all small reports as well as Delta files.
from pathlib import Path
import zipfile
from masar.workspace import completed_bronze_workspace
WORK = completed_bronze_workspace(ROOT)
pointer = ROOT / 'outputs/day01_bronze_success.json'
files_to_save = {pointer, *(p for p in WORK.rglob('*') if p.is_file())}
for name in ('source_inspection.json', 'cost_model_result.json'):
    files_to_save.update((ROOT / 'outputs').rglob(name))
archive = ROOT / 'outputs/day01_handoff.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(files_to_save):
        bundle.write(path, arcname=path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
    assert bundle.read('outputs/day01_bronze_success.json') == pointer.read_bytes()
    assert any(name.endswith('source_inspection.json') for name in bundle.namelist())
    assert any(name.endswith('cost_model_result.json') for name in bundle.namelist())
print('Keep this ZIP for the next day:', archive)
print('Also save this notebook with outputs and your LAB01/LAB02 notes.')
if IS_COLAB:
    from google.colab import files
    files.download(str(archive))

Keep this ZIP for the next day: /home/AlbandriAAlotaibi/masar-modern-data-engineering/outputs/day01_handoff.zip
Also save this notebook with outputs and your LAB01/LAB02 notes.


In [26]:
import os
os.environ['HADOOP_HOME'] = r'C:\hadoop'
os.environ['PATH'] = r'C:\hadoop\bin;' + os.environ['PATH']

In [27]:
from pathlib import Path
import json, sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'course.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook inside the complete course repository; see docs/SETUP.md.')
sys.path.insert(0, str(ROOT / 'src'))
SOURCE = ROOT / 'data/masar-small-v1'
from masar.workspace import require_fixed_dataset, completed_bronze_workspace
from masar.runtime import require_environment, start_spark
from masar.native_contracts import validate_stage_result
require_fixed_dataset(SOURCE)
require_environment()
WORK = completed_bronze_workspace(ROOT)
print('Continue workspace:', WORK.relative_to(ROOT))

Continue workspace: outputs/day01_bronze_lalzg7ih


In [28]:
from masar.silver import run_staging_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_staging_lab(spark, SOURCE, WORK)
    validate_stage_result('lab03a_staging', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print('Observed staging row counts:', result['counts'])
    preview = WORK / ('mini_lakehouse/staging/day02_' + result['run_id'] + '/stg_trips')
    spark.read.format('delta').load(str(preview)).select('trip_id', 'city', 'fare_sar').orderBy('trip_id').show(5, truncate=False)
finally:
    spark.stop()

{
  "scope": "DAY02_STAGING_ENGINE",
  "checks": {
    "counts_verified": true,
    "typed_values_match_source_oracle": true,
    "drivers_unique_and_join_safe": true,
    "gps_valid": true,
    "delta_readback": true
  }
}
Observed staging row counts: {'stg_trips': 144, 'stg_drivers': 6, 'stg_gps': 216}
+---------+------+--------+
|trip_id  |city  |fare_sar|
+---------+------+--------+
|SYN_T0001|Riyadh|18.00   |
|SYN_T0001|Riyadh|18.00   |
|SYN_T0002|Riyadh|19.25   |
|SYN_T0002|Riyadh|19.25   |
|SYN_T0003|Riyadh|20.50   |
+---------+------+--------+
only showing top 5 rows



In [29]:
import os
print("JAVA_HOME:", os.environ.get('JAVA_HOME'))

JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64


In [30]:
from masar.silver import run_incremental_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_incremental_lab(spark, SOURCE, WORK)
    validate_stage_result('lab03b_silver', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    spark.read.format('delta').load(str(WORK/'mini_lakehouse/silver/trips')).select('trip_id', 'city', 'fare_sar').orderBy('trip_id').show(5, truncate=False)
finally:
    spark.stop()

{
  "scope": "DAY02_SILVER_ENGINE",
  "checks": {
    "all_scenarios_match_independent_oracle": true,
    "business_keys_unique": true,
    "replay_preserves_business_content": true,
    "late_rows_retained": true,
    "actual_delta_files": true
  }
}
+-----------+------+--------+
|trip_id    |city  |fare_sar|
+-----------+------+--------+
|SYN_LATE001|Riyadh|25.00   |
|SYN_LATE002|Jeddah|27.00   |
|SYN_LATE003|Dammam|29.00   |
|SYN_T0001  |Riyadh|18.00   |
|SYN_T0002  |Riyadh|19.25   |
+-----------+------+--------+
only showing top 5 rows



In [31]:
from masar.dbt_lab import run_dbt_lab
dbt_report, dbt_path = run_dbt_lab(ROOT)
print(json.dumps({'status': dbt_report['status'], 'phases_completed': len(dbt_report['phases']), 'report': str(dbt_path.relative_to(ROOT)), 'error': dbt_report.get('error')}, indent=2))
assert dbt_report['status'] == 'PASSED_DBT_NATIVE', dbt_report.get('error')

# Observed learning output
for phase in dbt_report['phases']:
    print(phase['phase'], 'rows:', phase['rows'], 'fare SAR:', phase['total_fare_sar'])
print('Catalog evidence:', dbt_report['commands'][-1])

{
  "status": "PASSED_DBT_NATIVE",
  "phases_completed": 4,
  "report": "outputs/dbt_validation_4sr39hyt/reports/dbt_attempt.json",
  "error": null
}
base rows: 72 fare SAR: 1794.60
rerun rows: 72 fare SAR: 1794.60
late rows: 75 fare SAR: 1875.60
late_replay rows: 75 fare SAR: 1875.60
Catalog evidence: {'catalog_sha256': '2fcf976d5e9f444ef94d3703e6d5c525f9f9d1a4e116919434dfc5a18cd17dbd', 'models_documented': 6, 'sources_documented': 3, 'metadata_method': 'native DESCRIBE TABLE EXTENDED', 'phase': 'documentation', 'command': ['docs', 'generate'], 'target': 'dbt/commands/documentation/target'}


In [32]:
from pathlib import Path
import zipfile
pointer = ROOT / 'outputs/day01_bronze_success.json'
archive = ROOT / 'outputs/day02_handoff.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    dbt_workspace = dbt_path.parent.parent
    for p in sorted(dbt_workspace.rglob('*')):
        if p.is_file():
            bundle.write(p, p.relative_to(ROOT).as_posix())
    bundle.write(pointer, pointer.relative_to(ROOT).as_posix())
    for path in sorted(WORK.rglob('*')):
        if path.is_file():
            bundle.write(path, path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
print('Retain the notebook outputs, notes and', archive.relative_to(ROOT))

Retain the notebook outputs, notes and outputs/day02_handoff.zip


In [33]:
from pathlib import Path
import json, sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'course.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook inside the complete course repository; see docs/SETUP.md.')
sys.path.insert(0, str(ROOT / 'src'))
SOURCE = ROOT / 'data/masar-small-v1'
from masar.workspace import require_fixed_dataset, completed_bronze_workspace
from masar.runtime import require_environment, start_spark
from masar.native_contracts import validate_stage_result
require_fixed_dataset(SOURCE)
require_environment()
WORK = completed_bronze_workspace(ROOT)
print('Continue workspace:', WORK.relative_to(ROOT))

Continue workspace: outputs/day01_bronze_lalzg7ih


In [34]:
from masar.delta_lab import run_transactions_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_transactions_lab(spark, SOURCE, WORK)
    validate_stage_result('lab04a_transactions', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print(json.dumps({k: result[k] for k in ('before','after_correction','past_version_read')}, indent=2, default=str))
finally:
    spark.stop()

26/09/15 23:18:55 ERROR Utils: Aborting task
org.apache.spark.sql.delta.schema.DeltaInvariantViolationException: [DELTA_VIOLATE_CONSTRAINT_WITH_VALUES] CHECK constraint fare_nonnegative ((fare_sar IS NOT NULL) AND (fare_sar >= 0)) violated by row with values:
 - fare_sar : -5.00
	at org.apache.spark.sql.delta.schema.DeltaInvariantViolationException$.getConstraintViolationWithValuesException(InvariantViolationException.scala:78)
	at org.apache.spark.sql.delta.schema.DeltaInvariantViolationException$.apply(InvariantViolationException.scala:106)
	at org.apache.spark.sql.delta.schema.DeltaInvariantViolationException$.apply(InvariantViolationException.scala:117)
	at org.apache.spark.sql.delta.schema.DeltaInvariantViolationException.apply(InvariantViolationException.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$SpecificUnsafeProjection.CheckDeltaInvariant_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$SpecificUnsafeProjection.apply(Unkno

{
  "scope": "DAY03_TRANSACTIONS_ENGINE",
  "checks": {
    "native_correction_matches_source_expectation": true,
    "business_rows_stay_75": true,
    "replay_and_stale_delivery_preserve_values": true,
    "same_revision_conflict_rejected": true,
    "actual_prior_version_read": true,
    "mixed_valid_invalid_batch_rejected_atomically": true
  }
}
{
  "before": {
    "rows": 75,
    "business_digest": "0d16e2795620ae0c0f54d3fcd52a5b13fb2c2a47cd4d0c42c8ddb195f19bc6e0",
    "version": 1,
    "schema": [
      [
        "trip_id",
        "string"
      ],
      [
        "driver_id",
        "string"
      ],
      [
        "city",
        "string"
      ],
      [
        "start_utc",
        "timestamp"
      ],
      [
        "end_utc",
        "timestamp"
      ],
      [
        "trip_date_local",
        "date"
      ],
      [
        "fare_sar",
        "decimal(12,2)"
      ],
      [
        "distance_km",
        "decimal(12,2)"
      ],
      [
        "duration_seconds",

In [35]:
from masar.delta_lab import run_maintenance_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_maintenance_lab(spark, SOURCE, WORK)
    validate_stage_result('lab04b_maintenance', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print(json.dumps({'recovery': result['recovery'], 'vacuum': result['vacuum']}, indent=2, default=str))
finally:
    spark.stop()

{
  "scope": "DAY03_MAINTENANCE_ENGINE",
  "checks": {
    "unexpected_column_rejected": true,
    "approved_evolution_preserves_business_values": true,
    "compaction_preserves_values": true,
    "delete_affects_copy_only": true,
    "restore_creates_new_commit": true,
    "vacuum_is_non_destructive_dry_run": true,
    "trusted_silver_unchanged": true
  }
}
{
  "recovery": {
    "before": {
      "rows": 75,
      "business_digest": "1321d375742d806a1f0fef82be9e2862af04f5d18f3a976792a6b1d9aa1b4383",
      "version": 0,
      "schema": [
        [
          "trip_id",
          "string"
        ],
        [
          "driver_id",
          "string"
        ],
        [
          "city",
          "string"
        ],
        [
          "start_utc",
          "timestamp"
        ],
        [
          "end_utc",
          "timestamp"
        ],
        [
          "trip_date_local",
          "date"
        ],
        [
          "fare_sar",
          "decimal(12,2)"
        ],
       

In [36]:
from pathlib import Path
import zipfile
pointer = ROOT / 'outputs/day01_bronze_success.json'
archive = ROOT / 'outputs/day03_handoff.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(pointer, pointer.relative_to(ROOT).as_posix())
    for path in sorted(WORK.rglob('*')):
        if path.is_file():
            bundle.write(path, path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
print('Retain the notebook outputs, notes and', archive.relative_to(ROOT))

Retain the notebook outputs, notes and outputs/day03_handoff.zip


In [2]:
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ.get('PATH', '')

from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src" / "masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open from the repository root")
sys.path.insert(0, str(ROOT / "src"))

from masar.runtime import require_environment, start_spark
from masar.workspace import completed_bronze_workspace, require_fixed_dataset

SOURCE = ROOT / "data/masar-small-v1"
require_fixed_dataset(SOURCE)
require_environment()
WORK = completed_bronze_workspace(ROOT)
print("Ready:", WORK.relative_to(ROOT))


Ready: outputs/day01_bronze_lalzg7ih


In [38]:
IS_COLAB = False

In [45]:
from pyspark.sql import SparkSession
if SparkSession.getActiveSession():
    SparkSession.getActiveSession().stop()
    print("Cleaned up old Spark session")


In [46]:
import subprocess, sys
subprocess.run([
    sys.executable, "-c",
    "from pyspark.sql import SparkSession; s = SparkSession.builder.master('local[1]').appName('jar-dl')"
    ".config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.8')"
    ".getOrCreate(); s.stop()"
], check=True)
print("Done")


:: loading settings :: url = jar:file:/home/AlbandriAAlotaibi/masar-modern-data-engineering/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/AlbandriAAlotaibi/.ivy2/cache
The jars for the packages stored in: /home/AlbandriAAlotaibi/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-81d8abdd-4e02-4865-af3b-5359ed1b778c;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.8 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.8 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 740ms ::

Done


In [47]:
from masar.streaming import run_stream_lab
spark = start_spark(WORK, kafka=True)
try:
    result = run_stream_lab(spark, SOURCE, WORK)
    validate_stage_result('lab05_streaming', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print('Transport rows:', [phase['transport_rows'] for phase in result['phases']])
    print('Unique event IDs:', [phase['unique_event_ids'] for phase in result['phases']])
    spark.read.format('delta').load(str(WORK/result['event_table'])).select('event_id','trip_id','event_ts').orderBy('event_id').show(5, truncate=False)
finally:
    spark.stop()

ERROR:kafka.producer.sender:<Sender client_id=masar-9477d9309b3e transactional_id=None>: Uncaught error in kafka producer I/O thread
Traceback (most recent call last):
  File "/home/AlbandriAAlotaibi/masar-modern-data-engineering/.venv/lib/python3.11/site-packages/kafka/producer/sender.py", line 110, in run
    self.run_once()
  File "/home/AlbandriAAlotaibi/masar-modern-data-engineering/.venv/lib/python3.11/site-packages/kafka/producer/sender.py", line 149, in run_once
    self._maybe_wait_for_producer_id()
  File "/home/AlbandriAAlotaibi/masar-modern-data-engineering/.venv/lib/python3.11/site-packages/kafka/producer/sender.py", line 372, in _maybe_wait_for_producer_id
    response = self._client.send_and_receive(node_id, request)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/AlbandriAAlotaibi/masar-modern-data-engineering/.venv/lib/python3.11/site-packages/kafka/client_async.py", line 1191, in send_and_receive
    raise future.exception
kafka.errors.Can

AnalysisException: Failed to find data source: kafka. Please deploy the application as per the deployment section of Structured Streaming + Kafka Integration Guide.

In [6]:
import json

In [7]:
from masar.quality_gate import run_quality_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_quality_lab(spark, SOURCE, WORK)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print('Quarantined records:')
    spark.read.format('delta').load(str(WORK/result['quarantine_table'])).show(7, truncate=False)
    print('Approved rows:', spark.read.format('delta').load(str(WORK/result['approved_table'])).count())
finally:
    spark.stop()

/home/AlbandriAAlotaibi/masar-modern-data-engineering/.venv/lib/python3.11/site-packages/great_expectations/data_context/store/_store_backend.py:88: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parsed_store_backend_id = store_backend_id_file_parser.parseString(
Calculating Metrics: 100%|██████████| 128/128 [00:00<00:00, 668.21it/s]
/home/AlbandriAAlotaibi/masar-modern-data-engineering/.venv/lib/python3.11/site-packages/great_expectations/data_context/store/_store_backend.py:88: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parsed_store_backend_id = store_backend_id_file_parser.parseString(
Calculating Metrics: 100%|██████████| 128/128 [00:00<00:00, 609.12it/s]
/home/AlbandriAAlotaibi/masar-modern-data-engineering/.venv/lib/python3.11/site-packages/great_expectations/data_context/store/_store_backend.py:88: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parsed_store_backend_id = store_backend_id

{
  "scope": "DAY04_NATIVE_QUALITY",
  "checks": {
    "trusted_and_rechecked_pass_gx": true,
    "mixed_candidate_fails_gx": true,
    "native_mixed_counts": true,
    "native_reasons_match_reference": true,
    "failed_candidate_not_promoted": true,
    "quarantine_delta_readback": true,
    "approved_readback_same_business_contents": true,
    "source_silver_untouched": true,
    "data_docs_exist_for_all_three_cases": true
  }
}
Quarantined records:
+-------------+----------+-------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+---------------------------------------------------------------------+
|candidate_row|trip_id   |reason_codes       |raw_business_json                             

In [8]:
from pathlib import Path
import zipfile
pointer = ROOT / 'outputs/day01_bronze_success.json'
archive = ROOT / 'outputs/day04_handoff.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(pointer, pointer.relative_to(ROOT).as_posix())
    for path in sorted(WORK.rglob('*')):
        if path.is_file():
            bundle.write(path, path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
print('Retain the notebook outputs, notes and', archive.relative_to(ROOT))

Retain the notebook outputs, notes and outputs/day04_handoff.zip


In [9]:
from pathlib import Path
import json, sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'course.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook inside the complete course repository; see docs/SETUP.md.')
sys.path.insert(0, str(ROOT / 'src'))
SOURCE = ROOT / 'data/masar-small-v1'
from masar.workspace import require_fixed_dataset, completed_bronze_workspace
from masar.runtime import require_environment, start_spark
from masar.native_contracts import validate_stage_result
require_fixed_dataset(SOURCE)
require_environment()
WORK = completed_bronze_workspace(ROOT)
print('Continue workspace:', WORK.relative_to(ROOT))

Continue workspace: outputs/day01_bronze_lalzg7ih


In [10]:
from masar.serving import run_recovery_exercise
spark = start_spark(WORK, kafka=False)
try:
    result = run_recovery_exercise(spark, SOURCE, WORK)
    validate_stage_result('lab07_gold_recovery', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
finally:
    spark.stop()

{
  "scope": "DAY05_NATIVE_RECOVERY",
  "checks": {
    "injected_failure_observed": true,
    "previous_release_preserved": true,
    "rebuild_has_new_identity": true,
    "content_equal": true
  }
}


In [11]:
from masar.serving import run_serving_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_serving_lab(spark, SOURCE, WORK)
    validate_stage_result('lab08_serving', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print('BI totals:', json.dumps(result['bi_summary'], indent=2))
    from masar.serving import read_release
    _, observed_tables = read_release(spark, WORK)
    print('AI feature example:', observed_tables['ai.zone_hourly_features'][0])
    print('Future label example:', observed_tables['ai.zone_hourly_labels'][0])
finally:
    spark.stop()

{
  "scope": "DAY05_NATIVE_SERVING",
  "checks": {
    "gold.zone_hourly_demand_schema_and_keys": true,
    "gold.driver_daily_schema_and_keys": true,
    "bi.dim_zone_schema_and_keys": true,
    "bi.dim_driver_schema_and_keys": true,
    "bi.dim_date_schema_and_keys": true,
    "bi.fact_trips_schema_and_keys": true,
    "ai.zone_hourly_features_schema_and_keys": true,
    "ai.zone_hourly_labels_schema_and_keys": true,
    "fact_grain_75": true,
    "foreign_keys_valid": true,
    "gold_fact_totals_match": true,
    "events_aggregated_before_join": true,
    "group_grains_reconcile": true,
    "labels_not_fabricated": true,
    "feature_availability_checked": true,
    "feature_label_keys_aligned": true
  }
}
BI totals: [
  {
    "zone_key": "Z_DAMMAM",
    "trip_count": 25,
    "total_fare_sar": "670.40"
  },
  {
    "zone_key": "Z_JEDDAH",
    "trip_count": 25,
    "total_fare_sar": "625.20"
  },
  {
    "zone_key": "Z_RIYADH",
    "trip_count": 25,
    "total_fare_sar": "585.00"
  }

In [12]:
from pathlib import Path
import zipfile
pointer = ROOT / 'outputs/day01_bronze_success.json'
archive = ROOT / 'outputs/day05_handoff.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(pointer, pointer.relative_to(ROOT).as_posix())
    for path in sorted(WORK.rglob('*')):
        if path.is_file():
            bundle.write(path, path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
print('Retain the notebook outputs, notes and', archive.relative_to(ROOT))

Retain the notebook outputs, notes and outputs/day05_handoff.zip
